# 🏭 NOTEBOOK 09: PRODUCTION ACTIVE HITL PIPELINE (VẬN HÀNH THỰC CHIẾN)
*(Tương tác từng bước: Nạp ảnh thô ➔ Tách tập ➔ Xuất ZIP gán nhãn ➔ Tự động Ingest ZIP nhãn ➔ Huấn luyện tiếp sức ➔ Rào chắn dừng)*

> **Đặc điểm Production**:
> 1. **Zero Mock**: Chạy trên dữ liệu và nhãn thật 100%.
> 2. **Không config bằng tay**: Thiết lập trực tiếp tên model, task và raw labels ngay tại Cell 1.
> 3. **Tự động hóa file ZIP**: Chỉ cần thả file ZIP nhãn vào `inbox/`, hệ thống tự động giải nén, parse YOLO/COCO và nạp nhãn.
> 4. **Pre-labeling thông minh**: Tùy chọn xuất gợi ý nhãn có rào chắn độ tin cậy (>= 0.70) để tránh làm rối mắt annotator.


## ⚙️ Cell 1: Thiết Lập Cấu Hình Trực Tiếp (Không Cần Sửa YAML Thủ Công)


In [ ]:
import os
import sys
from pathlib import Path

# Thêm đường dẫn mã nguồn module 09
current_dir = Path.cwd().resolve()
if str(current_dir) not in sys.path:
    sys.path.insert(0, str(current_dir))

# ==============================================================================
# NHẬP THAM SỐ TRỰC TIẾP TẠI ĐÂY (TỰ ĐỘNG ĐỒNG BỘ CONFIG, KHÔNG CẦN SỬA YAML):
# ==============================================================================
MODEL_NAME = "yolo11n.pt"      # Tùy chọn: "yolo11n.pt", "yolo11n-seg.pt", "yolo11n-pose.pt", "checkpoint.pt"
TASK = "detect"                # Tùy chọn: "detect" (BBox), "segment" (Polygon), "pose" (Skeleton)

# Đường dẫn kho ảnh thô ban đầu (người dùng có thể trỏ tới bất kỳ thư mục ảnh nào)
RAW_IMAGES_DIR = "workspace/data/raw_images"

# [TÙY CHỌN]: Kho nhãn thô ban đầu (Thư mục hoặc file .zip có sẵn từ trước, đặt None nếu chưa có nhãn)
RAW_LABELS_PATH = None

# Ngưỡng tin cậy khi xuất nhãn gợi ý (Pre-labels)
PRELABEL_MIN_CONF = 0.70

# Ngưỡng mục tiêu mAP để dừng vòng lặp (Target Ceiling)
TARGET_mAP50 = 26.85

print("✓ Cấu hình đã nạp:")
print(f"  - Model: {MODEL_NAME} | Task: {TASK}")
print(f"  - Raw Images: {RAW_IMAGES_DIR}")
print(f"  - Raw Labels: {RAW_LABELS_PATH or 'Cold Start (Chưa có nhãn ban đầu)'}")
print(f"  - Pre-label Confidence Gate: >= {PRELABEL_MIN_CONF}")


## 🚀 Cell 2: Khởi Tạo Pipeline Điều Phối & Tự Động Nạp Raw Labels (Nếu Có)


In [ ]:
from src.pipeline import ProductionHITLPipeline

# Khởi tạo Pipeline điều phối trung tâm
pipeline = ProductionHITLPipeline(
    model_name=MODEL_NAME,
    task=TASK,
    raw_images_dir=RAW_IMAGES_DIR,
    raw_labels_path=RAW_LABELS_PATH,
    target_map50=TARGET_mAP50,
    prelabel_min_conf=PRELABEL_MIN_CONF,
)


## 📊 Cell 3: Phân Tách Tập Dữ Liệu Ban Đầu
Hệ thống tự động phân chia kho ảnh thô thành:
- `fixed_test/` (20% ảnh - Đóng băng dùng chung)
- `fixed_val/` (10% ảnh - Đóng băng)
- `seed_v0/` (Tập mầm ban đầu - ví dụ 100 ảnh)
- `unlabeled_pool/` (Phần còn lại để Active Mining qua các vòng)


In [ ]:
# Thực hiện phân tách tập dữ liệu (Tập mầm Seed V0 = 100 ảnh)
splits_info = pipeline.setup_initial_splits(seed_v0_size=100)


## 🛑 Cell 4: Kiểm Tra Nhãn Seed V0 & Xuất Gói Nhiệm Vụ Gán Nhãn
Nếu bạn đã nạp RAW_LABELS_PATH ở Cell 1, nhãn của seed_v0 đã được map tự động!
Nếu chưa có nhãn, hệ thống sẽ xuất file ZIP ra outbox/ và dừng lại chờ bạn nạp file ZIP nhãn vào inbox/.


In [ ]:
is_v0_ready = pipeline.check_seed_labels_ready(split_name="seed_v0")

if not is_v0_ready:
    print("⚠️ TẬP MẦM SEED V0 CHƯA CÓ NHÃN!")
    task_zip = pipeline.export_labeling_task(
        iteration=0,
        include_prelabels=False,  # Vòng 0 chưa có model huấn luyện -> xuất ảnh sạch thuần túy
    )
    print(f"👉 HÃY LẤY FILE NHIỆM VỤ ĐỂ GỬI ĐI GÁN NHÃN: {task_zip}")
    print(f"👉 SAU KHI GÁN NHÃN XONG, HÃY THẢ FILE ZIP NHÃN VÀO THƯ MỤC: {pipeline.inbox_dir}")
    print("👉 SAU ĐÓ CHẠY TIẾP CELL 5 ĐỂ TỰ ĐỘNG NẠP VÀ HUẤN LUYỆN MODEL V0!")
else:
    print("✅ Tập mầm seed_v0 ĐÃ CÓ ĐỦ NHÃN! Sẵn sàng huấn luyện Model V0.")


## 🔥 Cell 5: Tự Động Nạp File ZIP Nhãn Từ Inbox & Huấn Luyện Model V0
Cell này sẽ tự động tìm file .zip mới nhất bạn vừa thả vào thư mục inbox/, tự động giải nén, kiểm toán hợp lệ và kích hoạt huấn luyện Model V0!


In [ ]:
if not pipeline.check_seed_labels_ready(split_name="seed_v0"):
    print("Đang quét thư mục inbox/ để nạp file ZIP nhãn...")
    ingest_report = pipeline.ingest_labels_from_inbox(iteration=0)
    print(f"Kết quả nạp: {ingest_report.status} ({ingest_report.total_label_files} files nhãn)")
else:
    pipeline.consolidate_train_split(iteration=0)

# Huấn luyện Model V0 trên tập seed_v0 (15 epochs)
v0_metrics = pipeline.train_model(iteration=0, epochs=15)


## 🔍 Cell 6: Active Mining Vòng 1 (Khai Thác 100 Ảnh Khó Nhất) & Xuất Gói Pre-Labels
Model V0 sẽ quét toàn bộ kho ảnh chưa gán nhãn unlabeled_pool/, tính toán Uncertainty và bốc ra 100 ảnh thách thức nhất vào seed_v1/images/.
Hệ thống xuất kèm nhãn gợi ý sẵn (Pre-annotations) cho các đối tượng có độ tin cậy >= 0.70!


In [ ]:
# 1. Bốc 100 ảnh thách thức nhất vào seed_v1
mined_images = pipeline.mine_active_batch(current_iteration=0, batch_size=100)

# 2. Xuất gói gán nhãn có Pre-annotations lọc độ tin cậy cao
task_v1_zip = pipeline.export_labeling_task(
    iteration=1,
    include_prelabels=True,       # Bật nhãn gợi ý sẵn
    min_confidence=PRELABEL_MIN_CONF,  # Chỉ giữ box/poly có Conf >= 0.70
)

print(f"👉 ĐÃ XUẤT XƯỞNG GÓI NHIỆM VỤ VÒNG 1: {task_v1_zip}")
print("👉 Hãy tải gói này về nạp lên CVAT / Label Studio để duyệt và sửa nhãn nhanh.")
print(f"👉 Khi hoàn thành, thả file ZIP nhãn vào: {pipeline.inbox_dir} rồi chạy Cell 7!")


## 🔄 Cell 7: Nạp File ZIP Nhãn Vòng 1, Fine-tune Model V1 & Kiểm Tra Dừng
Chạy cell này sau khi bạn đã thả file ZIP nhãn vòng 1 vào thư mục inbox/.
Hệ thống tự động gộp tập huấn luyện tích lũy train_v1 (Seed V0 + Seed V1) và fine-tune tiếp sức kế thừa weights từ Model V0!


In [ ]:
# 1. Nạp nhãn vòng 1 từ inbox
report_v1 = pipeline.ingest_labels_from_inbox(iteration=1)

# 2. Huấn luyện tiếp sức Model V1 kế thừa từ Model V0 (chỉ 10-15 epochs)
v1_metrics = pipeline.train_model(iteration=1, epochs=15)

# 3. Kiểm tra rào chắn dừng
verdict_v1 = pipeline.stopping_checker.check(
    current_map50=v1_metrics["map50"],
    previous_map50=v0_metrics["map50"],
    current_budget_percent=20.0,
)

print(f"\n🏁 [BẢN ÁN RÀO CHẮN DỪNG]: {verdict_v1.status}")
print(f"  {verdict_v1.reason}")


## 🏆 Cell 8: Xuất Xưởng Checkpoint Production & Export ONNX / INT8 Cho Xe Tự Hành
Khi đạt mốc chất lượng mong muốn, cell này sẽ niêm phong checkpoint chính thức và xuất xưởng sang các định dạng tối ưu cạnh biên.


In [ ]:
# Xuất xưởng các định dạng Production
production_models = pipeline.export_production_model()

print("🎉 XUẤT XƯỞNG THÀNH CÔNG MODEL PRODUCTION:")
for fmt, path in production_models.items():
    print(f"  - [{fmt.upper()}]: {path}")
